# FinGuard Fraud Detection Pipeline
## Notebook 02 — Silver Layer Transformations

---

| | |
|---|---|
| **Layer** | Silver (Cleaned & Enriched) |
| **Source** | `finguard.bronze.*` |
| **Target** | `finguard.silver.*` (Delta Lake) |
| **Pattern** | Medallion Architecture — Silver Layer |
| **Context** | Australian retail banking — CBA / NAB / Westpac style |

---

### What This Notebook Does

The Silver layer is where **raw data becomes trustworthy data.**  
Rule: clean, deduplicate, enrich, and standardise — still no business aggregations (that's Gold).

| Step | Description |
|------|-------------|
| 1 | Read from Bronze Delta tables |
| 2 | Type casting and data standardisation |
| 3 | Deduplication (idempotent pipeline) |
| 4 | Null handling strategy |
| 5 | Broadcast joins — transactions + customers + merchants |
| 6 | Window functions — rolling spend, velocity, ranking |
| 7 | Dead-letter table — quarantine bad rows |
| 8 | Write to Silver Delta tables |
| 9 | Data quality checks |

## Cell 1 — Imports

In [ ]:
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    BooleanType,
    DateType,
    DoubleType,
    IntegerType,
    StringType,
    TimestampType,
)
from delta.tables import DeltaTable

print(f"Spark version   : {spark.version}")
print(f"Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} AEST")

## Cell 2 — Configuration

In [ ]:
# ── Unity Catalog identifiers ─────────────────────────────────────────────────
CATALOG_NAME   = "finguard"
BRONZE_SCHEMA  = "bronze"
SILVER_SCHEMA  = "silver"
MONITOR_SCHEMA = "monitoring"

# ── Source tables (Bronze) ────────────────────────────────────────────────────
BRONZE_TRANSACTIONS = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.transactions"
BRONZE_CUSTOMERS    = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.customers"
BRONZE_MERCHANTS    = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.merchants"

# ── Target tables (Silver) ────────────────────────────────────────────────────
SILVER_TRANSACTIONS_CLEANED  = f"{CATALOG_NAME}.{SILVER_SCHEMA}.transactions_cleaned"
SILVER_TRANSACTIONS_ENRICHED = f"{CATALOG_NAME}.{SILVER_SCHEMA}.transactions_enriched"
SILVER_CUSTOMERS             = f"{CATALOG_NAME}.{SILVER_SCHEMA}.customers"

# ── Monitoring tables ─────────────────────────────────────────────────────────
DEAD_LETTER_TABLE = f"{CATALOG_NAME}.{MONITOR_SCHEMA}.dead_letter"

# ── Batch ID ──────────────────────────────────────────────────────────────────
BATCH_ID      = f"silver_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
PIPELINE_NAME = "finguard_silver_transformation"

# ── AUSTRAC reporting threshold (AUD) ─────────────────────────────────────────
AUSTRAC_THRESHOLD = 10_000.00

print(f"Batch ID : {BATCH_ID}")
print(f"Source   : {CATALOG_NAME}.{BRONZE_SCHEMA}.*")
print(f"Target   : {CATALOG_NAME}.{SILVER_SCHEMA}.*")

## Cell 3 — Create Silver and Monitoring Schemas

In [ ]:
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}
    COMMENT 'FinGuard Silver layer — cleaned and enriched Delta tables'
""")

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{MONITOR_SCHEMA}
    COMMENT 'FinGuard pipeline monitoring — DQ results and dead-letter rows'
""")

print(f"✓ Schema ready: {CATALOG_NAME}.{SILVER_SCHEMA}")
print(f"✓ Schema ready: {CATALOG_NAME}.{MONITOR_SCHEMA}")

## Cell 4 — Read Bronze Tables

> **Interview point:** Always read from Delta tables using `spark.table()` rather than  
> `spark.read.format('delta').load(path)` when tables are registered in Unity Catalog.  
> It's cleaner, uses the metastore for path resolution, and respects access controls.

In [ ]:
# Read all three Bronze tables
# Spark is lazy — no data is actually read until an action is called
txn_bronze       = spark.table(BRONZE_TRANSACTIONS)
customers_bronze = spark.table(BRONZE_CUSTOMERS)
merchants_bronze = spark.table(BRONZE_MERCHANTS)

# Strip audit columns from dimension tables before joining
# We re-add Silver-specific audit columns after transformation
audit_cols = ["_ingested_at", "_source_file", "_batch_id", "_pipeline_name"]

txn_raw       = txn_bronze.drop(*audit_cols)
customers_raw = customers_bronze.drop(*audit_cols)
merchants_raw = merchants_bronze.drop(*audit_cols)

print(f"Bronze transactions : {txn_raw.count():>10,} rows")
print(f"Bronze customers    : {customers_raw.count():>10,} rows")
print(f"Bronze merchants    : {merchants_raw.count():>10,} rows")

## Cell 5 — Type Casting & Standardisation

> **Interview point:** Why do we cast types in Silver and not Bronze?  
> Bronze rule: land data exactly as received — never risk losing a row  
> because a type cast failed. In Silver we have a dead-letter table to  
> safely catch and quarantine rows that fail casting.

In [ ]:
# ── Cast transaction types ────────────────────────────────────────────────────
# Bronze landed txn_timestamp as StringType — cast to TimestampType here
# Bronze landed txn_date as StringType — cast to DateType here

txn_typed = (
    txn_raw
    # Timestamps and dates
    .withColumn(
        "txn_timestamp",
        F.to_timestamp(F.col("txn_timestamp"), "yyyy-MM-dd HH:mm:ss")
    )
    .withColumn(
        "txn_date",
        F.to_date(F.col("txn_date"), "yyyy-MM-dd")
    )
    # Standardise string fields to uppercase for consistent joins
    .withColumn("currency",      F.upper(F.col("currency")))
    .withColumn("channel",       F.lower(F.col("channel")))
    .withColumn("card_network",  F.initcap(F.col("card_network")))
    .withColumn("device_type",   F.lower(F.col("device_type")))
    .withColumn("ip_country",    F.upper(F.col("ip_country")))
    # Derived columns — useful for fraud feature engineering in Gold
    .withColumn("txn_year",      F.year(F.col("txn_timestamp")))
    .withColumn("txn_quarter",   F.quarter(F.col("txn_timestamp")))
    .withColumn("txn_week",      F.weekofyear(F.col("txn_timestamp")))
    .withColumn("is_weekend",    F.dayofweek(F.col("txn_timestamp")).isin([1, 7]))
    .withColumn("is_night",      F.col("txn_hour").between(22, 23) |
                                  F.col("txn_hour").between(0, 5))
    .withColumn(
        "amount_band",
        F.when(F.col("amount") < 50,    F.lit("micro"))
         .when(F.col("amount") < 200,   F.lit("small"))
         .when(F.col("amount") < 1_000, F.lit("medium"))
         .when(F.col("amount") < 5_000, F.lit("large"))
         .otherwise(F.lit("very_large"))
    )
    .withColumn(
        "is_near_austrac_threshold",
        F.col("amount").between(AUSTRAC_THRESHOLD * 0.90, AUSTRAC_THRESHOLD)
    )
)

# ── Cast customer types ───────────────────────────────────────────────────────
customers_typed = (
    customers_raw
    .withColumn("date_of_birth",     F.to_date(F.col("date_of_birth"), "yyyy-MM-dd"))
    .withColumn("account_open_date", F.to_date(F.col("account_open_date"), "yyyy-MM-dd"))
    .withColumn("address_state",     F.upper(F.col("address_state")))
    .withColumn("employment_status", F.lower(F.col("employment_status")))
    # Derived: customer age in years
    .withColumn(
        "age_years",
        F.floor(
            F.datediff(F.current_date(), F.col("date_of_birth")) / 365
        ).cast(IntegerType())
    )
    # Derived: account tenure in days
    .withColumn(
        "account_tenure_days",
        F.datediff(F.current_date(), F.col("account_open_date"))
    )
    # Income band for segmentation
    .withColumn(
        "income_band",
        F.when(F.col("annual_income_aud") < 50_000,  F.lit("low"))
         .when(F.col("annual_income_aud") < 100_000, F.lit("medium"))
         .when(F.col("annual_income_aud") < 200_000, F.lit("high"))
         .otherwise(F.lit("very_high"))
    )
)

print("✓ Type casting complete")
print(f"  Transaction columns : {len(txn_typed.columns)}")
print(f"  Customer columns    : {len(customers_typed.columns)}")

## Cell 6 — Deduplication

> **Interview point — very common question:**  
> *'How do you handle duplicate records in a PySpark pipeline?'*  
>  
> **Answer:** Use `dropDuplicates([primary_key])` not `distinct()`.  
> `distinct()` compares ALL columns — expensive and incorrect for dedup by key.  
> `dropDuplicates` lets you specify exactly which columns define uniqueness.  
> For transactions, we keep the row with the latest `_ingested_at` timestamp  
> using a Window function — this handles late-arriving duplicate feeds correctly.

In [ ]:
# ── Deduplication strategy ────────────────────────────────────────────────────
# A payment gateway may send the same transaction twice (retry on timeout).
# We keep the LATEST version of each transaction_id.
#
# Window: partition by transaction_id, order by txn_timestamp descending
# row_number() == 1 → the most recent record for each transaction_id

dedup_window = (
    Window
    .partitionBy("transaction_id")
    .orderBy(F.desc("txn_timestamp"))
)

txn_deduped = (
    txn_typed
    .withColumn("_row_num", F.row_number().over(dedup_window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

total_before = txn_typed.count()
total_after  = txn_deduped.count()
duplicates_removed = total_before - total_after

print(f"Before deduplication : {total_before:,}")
print(f"After deduplication  : {total_after:,}")
print(f"Duplicates removed   : {duplicates_removed:,}")

# Interview point:
# Why not just use .dropDuplicates(["transaction_id"]) ?
# dropDuplicates keeps an arbitrary row when duplicates exist.
# The Window approach lets us control WHICH duplicate to keep
# (latest, highest amount, specific source system, etc.)
# In banking: always keep the latest — it may have an updated response_code.

## Cell 7 — Null Handling Strategy

> **Interview point:**  
> *'How do you handle nulls in PySpark?'*  
>  
> Never blindly `fillna(0)` or `dropna()`. Each column needs a deliberate strategy:  
> - **Critical columns** (transaction_id, customer_id) → route to dead-letter table  
> - **Categorical columns** → fill with `'unknown'`  
> - **Numeric columns** → fill with `0` or median depending on context  
> - **Flag columns** → fill with `False`  
> - **Optional columns** → leave as null (preserve information)

In [ ]:
# ── Step 1: Separate bad rows → dead-letter table ─────────────────────────────
# Rows missing critical keys cannot be processed — quarantine them.
# In production these get investigated and reprocessed or escalated.

CRITICAL_COLUMNS = ["transaction_id", "customer_id", "merchant_id", "amount"]

# Build null condition for any critical column
null_condition = F.lit(False)
for col_name in CRITICAL_COLUMNS:
    null_condition = null_condition | F.col(col_name).isNull()

# Also flag negative amounts as bad rows
bad_rows = txn_deduped.filter(
    null_condition | (F.col("amount") <= 0)
).withColumn("_rejection_reason",
    F.when(F.col("transaction_id").isNull(), F.lit("null_transaction_id"))
     .when(F.col("customer_id").isNull(),    F.lit("null_customer_id"))
     .when(F.col("merchant_id").isNull(),    F.lit("null_merchant_id"))
     .when(F.col("amount").isNull(),         F.lit("null_amount"))
     .when(F.col("amount") <= 0,             F.lit("invalid_amount"))
     .otherwise(F.lit("unknown"))
).withColumn("_rejected_at",   F.current_timestamp())
 .withColumn("_batch_id",      F.lit(BATCH_ID))
 .withColumn("_pipeline_name", F.lit(PIPELINE_NAME))

bad_row_count = bad_rows.count()

# Write bad rows to dead-letter table
if bad_row_count > 0:
    (
        bad_rows.write
        .format("delta")
        .mode("append")
        .saveAsTable(DEAD_LETTER_TABLE)
    )
    print(f"⚠️  {bad_row_count:,} bad rows quarantined → {DEAD_LETTER_TABLE}")
else:
    print("✓ No bad rows found — dead-letter table empty")

# Keep only good rows for Silver
txn_clean = txn_deduped.filter(
    ~null_condition & (F.col("amount") > 0)
)
print(f"✓ Clean transactions: {txn_clean.count():,}")

In [ ]:
# ── Step 2: Fill non-critical nulls with sensible defaults ────────────────────

txn_clean = (
    txn_clean
    # Categorical — unknown is more honest than dropping the row
    .fillna({
        "channel":          "unknown",
        "card_network":     "unknown",
        "device_type":      "unknown",
        "ip_country":       "unknown",
        "merchant_category": "unknown",
        "mcc_code":         "9999",
        "response_code":    "00",
        "txn_day_of_week":  "unknown",
        "fraud_type":       "none",
        "fraud_indicator":  "none",
    })
    # Boolean flags — default False is safe for fraud/decline flags
    .fillna({
        "is_declined":     False,
        "is_international": False,
        "is_fraud":        False,
        "is_weekend":      False,
        "is_night":        False,
        "is_near_austrac_threshold": False,
    })
    # Numeric — 0 is safe for hour (midnight), not for amount (already filtered above)
    .fillna({"txn_hour": 0})
)

print("✓ Null filling complete")

## Cell 8 — Broadcast Joins

> **Interview point — extremely common:**  
> *'What is a broadcast join and when do you use it?'*  
>  
> **Answer:** A broadcast join sends the smaller DataFrame to every executor  
> so the join happens in memory without a shuffle.  
> Use when one side is small enough to fit in executor memory (< ~100MB rule of thumb).  
> Default broadcast threshold in Spark: 10MB (`spark.sql.autoBroadcastJoinThreshold`).  
>  
> In our case:  
> - transactions = 500,000 rows (~120MB) → large side  
> - customers    = 5,000 rows   (~1MB)   → broadcast ✓  
> - merchants    = 800 rows     (~120KB) → broadcast ✓  
>  
> Without broadcast: Spark does a sort-merge join = expensive shuffle across network.  
> With broadcast: join happens locally on each executor = 10-50x faster.

In [ ]:
# ── Prepare dimension tables for joining ──────────────────────────────────────
# Select only the columns needed for enrichment — avoid column name collisions

customers_dim = customers_typed.select(
    F.col("customer_id"),
    F.col("address_state").alias("customer_state"),
    F.col("address_suburb").alias("customer_suburb"),
    F.col("annual_income_aud"),
    F.col("employment_status"),
    F.col("credit_score"),
    F.col("age_years"),
    F.col("income_band"),
    F.col("account_tenure_days"),
    F.col("is_high_risk"),
    F.col("kyc_verified"),
)

merchants_dim = merchants_raw.select(
    F.col("merchant_id"),
    F.col("merchant_name"),
    F.col("category").alias("merchant_category_detail"),
    F.col("country").alias("merchant_country"),
    F.col("is_international").alias("merchant_is_international"),
    F.col("is_online_only").alias("merchant_is_online"),
    F.col("risk_level").alias("merchant_risk_level"),
    F.col("abn"),
)

print(f"customers_dim columns : {len(customers_dim.columns)}")
print(f"merchants_dim columns : {len(merchants_dim.columns)}")

In [ ]:
# ── Broadcast join: transactions + customers + merchants ──────────────────────
# F.broadcast() is an explicit hint to Spark — broadcast this DataFrame
# even if it's above the auto-broadcast threshold.
# Always use LEFT JOIN — don't drop transactions with no matching customer/merchant
# (referential integrity issues are common in real banking data).

txn_enriched = (
    txn_clean
    .join(
        F.broadcast(customers_dim),
        on="customer_id",
        how="left",
    )
    .join(
        F.broadcast(merchants_dim),
        on="merchant_id",
        how="left",
    )
)

# Fill nulls introduced by LEFT JOIN (customer/merchant not found in dim tables)
txn_enriched = txn_enriched.fillna({
    "customer_state":           "unknown",
    "customer_suburb":          "unknown",
    "income_band":              "unknown",
    "employment_status":        "unknown",
    "merchant_name":            "unknown",
    "merchant_risk_level":      "unknown",
    "merchant_country":         "unknown",
    "merchant_is_international": False,
    "merchant_is_online":        False,
    "is_high_risk":              False,
    "kyc_verified":              True,
})

print(f"✓ Enriched transactions: {txn_enriched.count():,} rows")
print(f"  Total columns after join: {len(txn_enriched.columns)}")

In [ ]:
# Preview enriched data
display(
    txn_enriched.select(
        "transaction_id", "customer_id", "amount", "channel",
        "customer_state", "income_band", "merchant_name",
        "merchant_risk_level", "is_fraud"
    ).limit(10)
)

## Cell 9 — Window Functions

> **Interview point — the most tested PySpark topic:**  
> *'Explain Window functions and give an example.'*  
>  
> **Answer:** A Window function performs a calculation across a set of rows  
> related to the current row, without collapsing them into a single output row  
> (unlike groupBy which reduces rows).  
>  
> Key components:  
> - `partitionBy` — like GROUP BY, defines the window boundary  
> - `orderBy`     — defines row order within the window  
> - `rowsBetween` / `rangeBetween` — defines the frame (which rows to include)  
>  
> In fraud detection: rolling spend, transaction count, and velocity features  
> are ALL calculated using window functions.

In [ ]:
# ── Window definitions ────────────────────────────────────────────────────────
# partitionBy customer_id so all calculations are per-customer
# orderBy txn_timestamp so rows are in chronological order

# Unbounded: from first transaction ever to current row
window_unbounded = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("txn_timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# Rolling 7-day window (in seconds: 7 * 24 * 60 * 60 = 604,800)
# rangeBetween works on the ORDER BY column value (timestamp as long/epoch)
window_7d = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("txn_timestamp").cast("long"))
    .rangeBetween(-604_800, 0)
)

# Rolling 30-day window (30 * 24 * 60 * 60 = 2,592,000)
window_30d = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("txn_timestamp").cast("long"))
    .rangeBetween(-2_592_000, 0)
)

# Ranking window — no frame, just order
window_rank = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("txn_timestamp").cast("long"))
)

print("✓ Window specs defined")
print("  window_unbounded : all prior transactions per customer")
print("  window_7d        : rolling 7-day per customer")
print("  window_30d       : rolling 30-day per customer")
print("  window_rank      : sequential ranking per customer")

In [ ]:
# ── Apply window functions ────────────────────────────────────────────────────

txn_windowed = (
    txn_enriched

    # ── Rolling spend features ─────────────────────────────────────────────
    # 7-day rolling total spend per customer
    .withColumn(
        "spend_7d_aud",
        F.sum("amount").over(window_7d)
    )
    # 30-day rolling total spend per customer
    .withColumn(
        "spend_30d_aud",
        F.sum("amount").over(window_30d)
    )
    # 7-day transaction count per customer
    .withColumn(
        "txn_count_7d",
        F.count("transaction_id").over(window_7d)
    )
    # 30-day transaction count per customer
    .withColumn(
        "txn_count_30d",
        F.count("transaction_id").over(window_30d)
    )
    # 7-day average transaction amount
    .withColumn(
        "avg_amount_7d",
        F.round(F.avg("amount").over(window_7d), 2)
    )

    # ── Velocity features ──────────────────────────────────────────────────
    # Time since the previous transaction (seconds) — velocity attack signal
    .withColumn(
        "prev_txn_timestamp",
        F.lag("txn_timestamp", 1).over(window_rank)
    )
    .withColumn(
        "seconds_since_prev_txn",
        F.when(
            F.col("prev_txn_timestamp").isNotNull(),
            F.col("txn_timestamp").cast("long") -
            F.col("prev_txn_timestamp").cast("long")
        ).otherwise(F.lit(None))
    )
    # Flag: transaction within 10 minutes of previous (velocity attack)
    .withColumn(
        "is_rapid_succession",
        F.col("seconds_since_prev_txn") < 600
    )

    # ── Ranking features ───────────────────────────────────────────────────
    # Transaction sequence number per customer (1 = first ever transaction)
    .withColumn(
        "customer_txn_sequence",
        F.row_number().over(window_rank)
    )
    # Is this the customer's first transaction ever?
    .withColumn(
        "is_first_transaction",
        F.col("customer_txn_sequence") == 1
    )
    # Amount deviation from customer's 30-day average
    .withColumn(
        "amount_vs_30d_avg",
        F.round(
            F.col("amount") / F.nullif(F.avg("amount").over(window_30d), 0),
            2
        )
    )

    # ── Clean up intermediate columns ──────────────────────────────────────
    .drop("prev_txn_timestamp")
)

print(f"✓ Window functions applied")
print(f"  Total columns: {len(txn_windowed.columns)}")

In [ ]:
# Preview window function results for a single customer
sample_customer = (
    txn_windowed
    .orderBy("customer_id", "txn_timestamp")
    .select(
        "customer_id", "txn_timestamp", "amount",
        "spend_7d_aud", "txn_count_7d", "avg_amount_7d",
        "seconds_since_prev_txn", "is_rapid_succession",
        "customer_txn_sequence", "amount_vs_30d_avg"
    )
    .limit(20)
)
display(sample_customer)

## Cell 10 — Add Silver Audit Columns & Write

In [ ]:
# ── Add Silver audit columns ──────────────────────────────────────────────────

def add_silver_audit_columns(df, batch_id, pipeline_name):
    """Add Silver-layer audit columns.

    Interview point:
        Silver audit columns differ from Bronze.
        Bronze records when the raw file was ingested.
        Silver records when transformation was applied.
        Both are needed for full pipeline lineage.
    """
    return (
        df
        .withColumn("_silver_processed_at", F.current_timestamp())
        .withColumn("_silver_batch_id",     F.lit(batch_id))
        .withColumn("_pipeline_name",       F.lit(pipeline_name))
    )


txn_final = add_silver_audit_columns(txn_windowed, BATCH_ID, PIPELINE_NAME)

print("✓ Audit columns added")

In [ ]:
# ── Write Silver transactions (cleaned) ───────────────────────────────────────
# transactions_cleaned: deduped, typed, null-handled — no joins yet
# Useful for teams that want clean raw data without the enrichment columns

txn_clean_final = add_silver_audit_columns(txn_clean, BATCH_ID, PIPELINE_NAME)

(
    txn_clean_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("txn_month")
    .saveAsTable(SILVER_TRANSACTIONS_CLEANED)
)

count = spark.table(SILVER_TRANSACTIONS_CLEANED).count()
print(f"✓ {SILVER_TRANSACTIONS_CLEANED}: {count:,} rows")

In [ ]:
# ── Write Silver transactions (enriched) ──────────────────────────────────────
# transactions_enriched: full join + window features — Gold layer reads from here

(
    txn_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("txn_month")
    .saveAsTable(SILVER_TRANSACTIONS_ENRICHED)
)

count = spark.table(SILVER_TRANSACTIONS_ENRICHED).count()
print(f"✓ {SILVER_TRANSACTIONS_ENRICHED}: {count:,} rows")

In [ ]:
# ── Write Silver customers ────────────────────────────────────────────────────

customers_final = add_silver_audit_columns(
    customers_typed, BATCH_ID, PIPELINE_NAME
)

(
    customers_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("address_state")
    .saveAsTable(SILVER_CUSTOMERS)
)

count = spark.table(SILVER_CUSTOMERS).count()
print(f"✓ {SILVER_CUSTOMERS}: {count:,} rows")

## Cell 11 — Data Quality Checks

In [ ]:
silver_df = spark.table(SILVER_TRANSACTIONS_ENRICHED)
total     = silver_df.count()

print("Silver Layer — Data Quality Report")
print("═" * 50)
print(f"  Total rows             : {total:,}")

# 1. No nulls on critical columns
for col_name in CRITICAL_COLUMNS:
    null_count = silver_df.filter(F.col(col_name).isNull()).count()
    status = "✓" if null_count == 0 else "⚠️"
    print(f"  {status} Nulls in {col_name:<25}: {null_count:,}")

# 2. No negative amounts
neg_amounts = silver_df.filter(F.col("amount") <= 0).count()
print(f"  {'✓' if neg_amounts == 0 else '⚠️'} Negative amounts           : {neg_amounts:,}")

# 3. Window features populated
null_window = silver_df.filter(F.col("spend_7d_aud").isNull()).count()
print(f"  {'✓' if null_window == 0 else '⚠️'} Null spend_7d_aud          : {null_window:,}")

# 4. Fraud rate preserved from Bronze
fraud_rate = silver_df.filter(F.col("is_fraud") == True).count() / total * 100
print(f"\n  Fraud rate             : {fraud_rate:.2f}%")

# 5. Rapid succession transactions
rapid = silver_df.filter(F.col("is_rapid_succession") == True).count()
print(f"  Rapid succession txns  : {rapid:,}  ({rapid/total*100:.2f}%)")

# 6. AUSTRAC near-threshold
near_threshold = silver_df.filter(
    F.col("is_near_austrac_threshold") == True
).count()
print(f"  Near AUSTRAC threshold : {near_threshold:,}")

print("═" * 50)

## Cell 12 — OPTIMIZE Silver Tables

In [ ]:
print("Running OPTIMIZE on Silver tables...")

spark.sql(f"""
    OPTIMIZE {SILVER_TRANSACTIONS_ENRICHED}
    ZORDER BY (customer_id, txn_date, is_fraud)
""")

print(f"✓ OPTIMIZE complete: {SILVER_TRANSACTIONS_ENRICHED}")

# Interview point:
# Why ZORDER by is_fraud in Silver?
# Gold layer fraud queries will always filter WHERE is_fraud = TRUE.
# ZORDERing on is_fraud co-locates all fraud rows in fewer files
# so Gold reads only those files — massive I/O reduction.

## Cell 13 — Summary

In [ ]:
print("═" * 60)
print("  SILVER LAYER COMPLETE")
print("═" * 60)
print(f"  Batch ID : {BATCH_ID}")
print()

silver_tables = [
    SILVER_TRANSACTIONS_CLEANED,
    SILVER_TRANSACTIONS_ENRICHED,
    SILVER_CUSTOMERS,
]

for table in silver_tables:
    count = spark.table(table).count()
    print(f"  {table:<50} {count:>10,} rows")

print()
print("  Transformations applied:")
print("    ✓ Type casting (StringType → TimestampType, DateType)")
print("    ✓ Deduplication via Window row_number()")
print("    ✓ Dead-letter table for bad rows")
print("    ✓ Null handling strategy per column type")
print("    ✓ Broadcast joins (customers + merchants)")
print("    ✓ Window functions (7d/30d rolling, velocity, ranking)")
print("    ✓ Derived columns (amount_band, is_night, is_weekend)")
print("    ✓ OPTIMIZE + ZORDER BY (customer_id, txn_date, is_fraud)")
print()
print("  Next → notebooks/03_gold_feature_engineering.ipynb")
print("═" * 60)

---
## Interview Question Reference

| Question | Cell |
|----------|------|
| Why cast types in Silver and not Bronze? | Cell 5 |
| What is a broadcast join and when to use it? | Cell 8 |
| What is the difference between repartition and coalesce? | Cell 10 |
| How do you handle duplicates in PySpark? | Cell 6 |
| Why use dropDuplicates over distinct? | Cell 6 |
| What is a dead-letter table? | Cell 7 |
| How do you handle nulls in PySpark? | Cell 7 |
| What are Window functions? Explain partitionBy and orderBy. | Cell 9 |
| What is rowsBetween vs rangeBetween? | Cell 9 |
| What is lag() and lead()? | Cell 9 |
| What is row_number() vs rank() vs dense_rank()? | Cell 9 |
| How do you calculate rolling averages in PySpark? | Cell 9 |
| What is a velocity feature in fraud detection? | Cell 9 |
| Why ZORDER by is_fraud in Silver? | Cell 12 |